# Project: Super Store Data Cleaning and Analysis

**Objective:** Analyze and clean Super Store retail data in PostgreSQL to identify top-performing products and estimate missing order quantities using available sales, pricing, market, region, and discount information.

---
**Business Questions:**

1. Which products generate the highest total sales and profit within each product category?
2. How can missing order quantities be estimated using historical order data and product-level pricing information?
---
**Tables Used:**

* `orders`

  * `row_id` : Unique record identifier
  * `order_id` : Identifier for each order
  * `order_date` : Date the order was placed
  * `market` : Market associated with the order
  * `region` : Customer region
  * `product_id` : Identifier of the purchased product
  * `sales` : Total sales amount for the line item
  * `quantity` : Total quantity purchased for the line item
  * `discount` : Discount applied to the line item
  * `profit` : Total profit earned on the line item

**<br>**

* `products`

  * `product_id` : Unique product identifier
  * `category` : Product category
  * `sub_category` : Product sub-category
  * `product_name` : Detailed product name
---
**Methodology:**

1. Join the `orders` and `products` tables using `product_id` to connect order activity with product categories and names.
2. Aggregate sales and profit for each product to determine total product performance.
3. Round total sales and profit to two decimal places for reporting.
4. Use the `DENSE_RANK()` window function with `PARTITION BY category` to rank products independently within each category based on total sales.
5. Filter the results to retain the top five ranked products in each category.
6. Identify orders where `quantity` is missing and separate them from orders with known quantities.
7. Use completed orders to calculate unit price by dividing `sales` by `quantity`.
8. Match records with missing quantities to available pricing information using `product_id`, `discount`, `market`, and `region`.
9. Estimate missing quantities by dividing the order's sales amount by the corresponding unit price.
10. Round calculated quantities for reporting and return the original order information alongside the imputed quantity.

---
**Output Columns:**

**Analysis 1 — Top Five Products by Category**

* `category` : Product category
* `product_name` : Name of the product
* `product_total_sales` : Total sales generated by the product
* `product_total_profit` : Total profit generated by the product
* `product_rank` : Product's sales rank within its category

**Analysis 2 — Missing Quantity Imputation**

* `product_id` : Product identifier
* `discount` : Discount applied to the order
* `market` : Market associated with the order
* `region` : Customer region
* `sales` : Sales amount for the order line
* `quantity` : Original quantity, including missing values
* `calculated_quantity` : Estimated quantity based on the calculated unit price
---
**SQL Techniques Used:**

* Common Table Expressions (`WITH`)
* `INNER JOIN`
* `GROUP BY`
* `SUM()`
* `ROUND()`
* `DENSE_RANK()`
* Window functions
* `PARTITION BY`
* `ORDER BY`
* `WHERE`
* `IS NULL` and `IS NOT NULL`
* `DISTINCT`
* Arithmetic calculations for unit-price estimation
* Data imputation
* Relational joins using multiple matching conditions
* PostgreSQL type casting with `::NUMERIC`


In [5]:
-- 1. What are the top 5 products in each category based on total sales?
-- Create a CTE to calculate total sales and profit for each product
WITH ord AS (
	SELECT 
	p.category,
	p.product_name,
	-- Calculate total sales and total profit for each product
	ROUND(SUM(o.sales)::NUMERIC, 2) AS product_total_sales,
	ROUND(SUM(o.profit)::NUMERIC, 2) AS product_total_profit
FROM orders AS o
INNER JOIN products AS p
ON o.product_id = p.product_id
-- Group orders by category and product name	
GROUP BY p.category, p.product_name
ORDER BY p.category ASC
	),
-- Create another CTE to rank products within each category
rkd AS (
	SELECT 
	category,
	product_name,
	product_total_sales,
	product_total_profit,
	-- Use DENSE_RANK() so tied products get the same rank without gaps
	DENSE_RANK() 
	OVER(
		PARTITION BY category 
		ORDER BY product_total_sales DESC
		) AS product_rank
	FROM ord
	)
-- Get the results for the top 5 products in each category
SELECT * 
FROM rkd
WHERE product_rank < 6

,category,product_name,product_total_sales,product_total_profit,product_rank
0,Furniture,"Hon Executive Leather Armchair, Adjustable",58193.48,5997.25,1
1,Furniture,"Office Star Executive Leather Armchair, Adjust...",51449.80,4925.80,2
2,Furniture,"Harbour Creations Executive Leather Armchair, ...",50121.52,10427.33,3
3,Furniture,"SAFCO Executive Leather Armchair, Black",41923.53,7154.28,4
4,Furniture,"Novimex Executive Leather Armchair, Adjustable",40585.13,5562.35,5
5,Office Supplies,"Eldon File Cart, Single Width",39873.23,5571.26,1
6,Office Supplies,"Hoover Stove, White",32842.60,-2180.63,2
7,Office Supplies,"Hoover Stove, Red",32644.13,11651.68,3
8,Office Supplies,"Rogers File Cart, Single Width",29558.82,2368.82,4
9,Office Supplies,"Smead Lockers, Industrial",28991.66,3630.44,5


In [6]:
-- 2. Calculate quantity for orders with missing values.
-- Create a CTE to find orders with missing quantity values.
WITH unsolved AS(
	SELECT
	product_id,
	market,
	region,
	discount,
	quantity,
	sales
FROM orders
-- Filter for orders with missing quantities
WHERE quantity IS NULL
	),

-- Create a CTE to calculate unit price
-- from orders with quantity values
solved AS (
	SELECT
	product_id,
	market,
	region,
	discount,
	quantity,
	ROUND((sales/quantity)::NUMERIC, 2) AS unit_price
FROM orders
-- Keep orders with quantity values
WHERE quantity IS NOT NULL
	)

SELECT DISTINCT 
	-- SELECT all columns from the unsolved CTE
	u.*, 
	s.unit_price,
	-- Divide the sales values in the unsolved CTE by the
	-- unit price in the solved CTE to get missing quantities
	ROUND((u.sales/s.unit_price)::NUMERIC, 2) AS calculated_quantity
FROM unsolved AS u
-- Match each order to its corresponding unit price
INNER JOIN solved AS s 
ON u.product_id = s.product_id
	AND u.discount = s.discount
	AND u.market = s.market
	AND u.region = s.region

,product_id,market,region,discount,quantity,sales,unit_price,calculated_quantity
0,FUR-ADV-10000571,EMEA,EMEA,0.00,NaN,438.960,109.74,4.0
1,FUR-ADV-10004395,EMEA,EMEA,0.00,NaN,84.120,42.06,2.0
2,FUR-BO-10001337,US,West,0.15,NaN,308.499,102.83,3.0
3,TEC-STA-10003330,Africa,Africa,0.00,NaN,506.640,253.32,2.0
4,TEC-STA-10004542,Africa,Africa,0.00,NaN,160.320,40.08,4.0
